In [1]:
import pandas as pd
import random
import numpy as np
import torch
def set_seed(seed):
    """Set seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [2]:
dfCovid = pd.read_csv('data/dataRwaCovid.csv')

In [3]:
dfDiabetes = pd.read_csv('data/dataIHADiabetes.csv')

In [4]:
DiabetesTopics = [
    "Physical",
    "Psychological",
    "No Symptoms",
    "Prognosis",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Testing/Monitoring Devices",
    "Health Data",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Adverse Events",
    "Therapeutic Devices",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Insurance/Billing",
    "Medical Records",
    "Referrals",
    "Transportation",
    "Primary (Pharmaceutical Prevention)",
    "Primary (Non-Pharmaceutical Prevention)",
    "Secondary (Pharmaceutical Prevention)",
    "Secondary (Non-Pharmaceutical Prevention)",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Entertainment",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety Concerns",
    "Health Education",
    "Sexual & Reproductive Health",
    "Child & Family Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "Rapport",
    "Transition to Adult Clinic"
]

In [5]:
CovidTopics = [
    "Physical",
    "Mental/Emotional",
    "No Symptoms",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Pharmaceutical Prevention",
    "Non-Pharmaceutical Prevention",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety concern",
    "Health Education",
    "Maternal & Child Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "wave",
    "batch"
]

In [6]:
commonSubtopics = sorted(list(set(DiabetesTopics) & set(CovidTopics)))

In [7]:
dfCovid[commonSubtopics].sum()

,0
Alternative,35
Clinical,5
Counseling,0
Cultural/Religion,154
Diagnostic Methods - Other,5
Diet/Nutrition,102
Emergent,138
Exercise,25
Financial,53
Friends & Family,138


In [8]:
diabetesColumns = commonSubtopics[:]
diabetesColumns.insert(0,"conversation")
dfDiabetesNew = dfDiabetes[diabetesColumns]


In [9]:
covidColumns = commonSubtopics[:]
covidColumns.insert(0,"conversation(english_only)")
dfCovidNew = dfCovid[covidColumns]


In [10]:
dfCovidNew = dfCovidNew.rename(columns={"conversation(english_only)":"conversation"})

df

In [11]:
dfCombined = pd.concat([dfCovidNew, dfDiabetesNew])

In [12]:
dfCombined.sum()

,0
conversation,"(S): Hello, how are you today? Do you have any..."
Alternative,35
Clinical,6
Counseling,0
Cultural/Religion,343
Diagnostic Methods - Other,8
Diet/Nutrition,340
Emergent,140
Exercise,131
Financial,53


In [13]:
n = 100
label_sum = dfCombined[commonSubtopics].sum()
label_keep = label_sum[label_sum >= n].index.tolist()
label_keep_original = label_keep[:]
label_keep.insert(0,"conversation")

dfCombined_filtered = dfCombined[label_keep]
dfCombined_filtered.shape


(3907, 20)

In [14]:
dfCombined_filtered.sum()

,0
conversation,"(S): Hello, how are you today? Do you have any..."
Cultural/Religion,343
Diet/Nutrition,340
Emergent,140
Exercise,131
Friends & Family,291
Grateful Patient,291
Health Education,322
Laboratory/Testing,1078
Medications,418


In [15]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].str.lower()

/tmp/ipykernel_1133/1116881674.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].str.lower()


In [16]:
import nltk
import re
import string

In [17]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.6 MB/s eta 0:00:00


In [18]:
import contractions

In [19]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([contractions.fix(word) for word in x.split()]))

/tmp/ipykernel_1133/2646981620.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([contractions.fix(word) for word in x.split()]))


In [20]:
dfCombined_filtered.shape

(3907, 20)

In [21]:
dfCombined_filtered.head()

,conversation,Cultural/Religion,Diet/Nutrition,Emergent,Exercise,Friends & Family,Grateful Patient,Health Education,Laboratory/Testing,Medications,No Symptoms,Non-urgent,Outpatient Logistics/Scheduling,Physical,Physical Environment/Climate,Service Complaint,Social Services,Technical/IT,Urgent,Work/School
0,"(s): hello, how are you today? do you have any...",1,0,0,0,0,1,0,1,0,1,1,0,0,0,0,0,0,0,0
1,(s): welcome to rwanda ministry of health plat...,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0
2,(s): welcome to rwanda ministry of health plat...,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0
3,"(s): hello, how are you today? do you have any...",0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0
4,(s): welcome to rwanda ministry of health plat...,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0


In [22]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'\d+','',x))

/tmp/ipykernel_1133/3892837535.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'\d+','',x))


In [23]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'[.]',' ',x))

/tmp/ipykernel_1133/1879545715.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'[.]',' ',x))


In [24]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: "".join([char for char in x if char not in string.punctuation]))

/tmp/ipykernel_1133/2885321218.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: "".join([char for char in x if char not in string.punctuation]))


In [25]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [26]:
from nltk.corpus import stopwords

In [27]:
sw = stopwords.words('english')

In [28]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([word for word in x.split() if word not in sw]))

/tmp/ipykernel_1133/280526503.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([word for word in x.split() if word not in sw]))


In [29]:
import nltk
nltk.download("punkt_tab")
nltk.download('punkt')
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download('averaged_perceptron_tagger')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

from nltk.corpus import wordnet

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_with_pos(text):
    tokens = nltk.word_tokenize(text)
    tagged_tokens = nltk.pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in tagged_tokens])


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [30]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lemmatize_with_pos)

/tmp/ipykernel_1133/306948834.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lemmatize_with_pos)


In [31]:
dfCombined_filtered.head()
dfCombined_filtered.shape

(3907, 20)

In [32]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

dfCombined_filtered['conversation_tokens'] = dfCombined_filtered['conversation'].apply(word_tokenize)



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/tmp/ipykernel_1133/4092647881.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered['conversation_tokens'] = dfCombined_filtered['conversation'].apply(word_tokenize)


In [33]:
!pip install scikit-multilearn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 4.7 MB/s eta 0:00:00


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
from skmultilearn.model_selection import iterative_train_test_split
from sklearn.model_selection import train_test_split


In [35]:
tf = TfidfVectorizer(max_features=5000,ngram_range=(1,2),min_df=3,max_df=0.6 )
x =dfCombined_filtered["conversation"]

In [36]:
import torch
# from sklearn.multi
# torch.cuda.get_device_name()

In [37]:
len(label_keep_original)

y = dfCombined_filtered[label_keep_original].to_numpy()
print(y.shape)

(3907, 19)


In [38]:

import numpy as np

split = np.load("data/shared_split_indices.npz")
train_idx = split["train_idx"]
val_idx   = split["val_idx"]
test_idx  = split["test_idx"]

x_raw = dfCombined_filtered["conversation"].to_numpy()
y     = dfCombined_filtered[label_keep_original].to_numpy()

x_train_raw = x_raw[train_idx]
x_val_raw   = x_raw[val_idx]
x_test_raw  = x_raw[test_idx]

y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

df_aug_train = pd.read_csv("data/backtranslated_train.csv")

import pandas as pd
import numpy as np
import re
import string
import contractions
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

df_aug_train = pd.read_csv('data/backtranslated_train.csv')

y_train = df_aug_train[label_keep_original].to_numpy()

sw = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def lemmatize_with_pos(text):
    tokens = word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(tok, get_wordnet_pos(pos)) for tok, pos in tagged])

def preprocess(text):
    text = text.lower()
    text = " ".join([contractions.fix(word) for word in text.split()])
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[.]', ' ', text)
    text = "".join([c for c in text if c not in string.punctuation])
    text = " ".join([w for w in text.split() if w not in sw])
    text = lemmatize_with_pos(text)
    return text

df_aug_train['conversation'] = df_aug_train['conversation'].apply(preprocess)
x_train = df_aug_train["conversation"].to_numpy()

y_train = df_aug_train[label_keep_original].to_numpy()

x_train = tf.fit_transform(pd.Series(x_train.flatten()))
x_val   = tf.transform(pd.Series(x_val_raw.flatten()))
x_test  = tf.transform(pd.Series(x_test_raw.flatten()))

In [39]:
x_train.shape
type(x_train)


scipy.sparse._csr.csr_matrix

In [40]:
y_train.shape
type(y_train)

numpy.ndarray

In [41]:
from xgboost import XGBClassifier
from sklearn.multiclass import OneVsRestClassifier


In [42]:
# x_val,  y_val,  x_test,y_test = iterative_train_test_split(x_, y_, test_size=0.5)


In [43]:
!pip install imbalanced-learn

In [44]:
!pip install --upgrade scikit-multilearn


In [45]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 18.2 MB/s eta 0:00:00


In [46]:
import optuna

In [47]:
from sklearn.multiclass import OneVsRestClassifier
import numpy as np
from sklearn.metrics import f1_score

In [48]:
from cuml.ensemble import RandomForestClassifier

In [49]:
x_train = x_train.toarray()
x_val = x_val.toarray()
x_test = x_test.toarray()

In [50]:
from enum import auto
models = {}
list_params = {}
for i,label_name in enumerate(label_keep_original):
    print(label_name)
    def objective(trial):
      params = {
       'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
       'max_depth': trial.suggest_int('max_depth', 1, 10),
       'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
       'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 6),
       'n_bins': trial.suggest_int('n_bins', 10, 256),
      'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
      'split_criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
      'random_state': 42,

      }
      model = RandomForestClassifier(**params)
      print(type(x_train))
      print(type(y_train[:,i]))
      model.fit(x_train, y_train[:,i])
      x_val_predic = model.predict_proba(x_val)[:, 1]


      best_f1 = 0
      best_thresh = 0.5
      for threshold_val in np.arange(0.05, 0.96, 0.05):
          preds = (x_val_predic > threshold_val).astype(int)
          current_f1 = f1_score(y_val[:, i], preds)
          if current_f1 > best_f1:
              best_f1 = current_f1
              best_thresh = threshold_val
      return best_f1
    study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
    study.optimize(objective, n_trials=50)
    print(f"Best trial for {label_name}: F1 score= {study.best_value:.4f}")
    print(f"Best hyperparameters: {study.best_params}")
    list_params[label_name] = study.best_params

[I 2026-05-25 15:22:57,020] A new study created in memory with name: no-name-80467046-eeaf-4b40-a935-086d74ea3040


Cultural/Religion
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


[I 2026-05-25 15:23:03,296] Trial 0 finished with value: 0.912621359223301 and parameters: {'n_estimators': 674, 'max_depth': 4, 'min_samples_split': 8, 'min_samples_leaf': 2, 'n_bins': 126, 'max_features': 'sqrt', 'criterion': 'gini'}. Best is trial 0 with value: 0.912621359223301.


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


[I 2026-05-25 15:23:07,240] Trial 1 finished with value: 0.9038461538461539 and parameters: {'n_estimators': 377, 'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 2, 'n_bins': 250, 'max_features': 'sqrt', 'criterion': 'gini'}. Best is trial 0 with value: 0.912621359223301.


...

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


[I 2026-05-25 15:57:42,170] Trial 48 finished with value: 0.7 and parameters: {'n_estimators': 830, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 4, 'n_bins': 189, 'max_features': 'sqrt', 'criterion': 'gini'}. Best is trial 23 with value: 0.7317073170731707.


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


[I 2026-05-25 15:57:45,258] Trial 49 finished with value: 0.6818181818181818 and parameters: {'n_estimators': 909, 'max_depth': 7, 'min_samples_split': 6, 'min_samples_leaf': 4, 'n_bins': 118, 'max_features': 'sqrt', 'criterion': 'gini'}. Best is trial 23 with value: 0.7317073170731707.


Best trial for Work/School: F1 score= 0.7317
Best hyperparameters: {'n_estimators': 926, 'max_depth': 6, 'min_samples_split': 7, 'min_samples_leaf': 3, 'n_bins': 142, 'max_features': 'sqrt', 'criterion': 'gini'}


In [51]:
import numpy as np
from sklearn.metrics import f1_score

In [52]:
models = {}
for i,label_name in enumerate(label_keep_original):


  model = RandomForestClassifier(**list_params[label_name])

  model.fit(x_train, y_train[:,i])
  models[label_name] = model




In [53]:
optimal_thresholds = {}
import numpy as np
for i, label_name in enumerate(label_keep_original):

    x_val_predic = models[label_name].predict_proba(x_val)[:, 1]
    best_f1 = 0
    best_thresh = 0.5 # Default
    for threshold_val in np.arange(0.05, 0.96, 0.05):
        preds = (x_val_predic > threshold_val).astype(int)
        current_f1 = f1_score(y_val[:, i], preds)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_thresh = threshold_val
    optimal_thresholds[label_name] = best_thresh
    print(f"Optimal threshold for {label_name}: {best_thresh:.2f} (F1: {best_f1:.4f})")


Optimal threshold for Cultural/Religion: 0.15 (F1: 0.9293)
Optimal threshold for Diet/Nutrition: 0.15 (F1: 0.8627)
Optimal threshold for Emergent: 0.10 (F1: 0.4706)
Optimal threshold for Exercise: 0.10 (F1: 0.8571)
Optimal threshold for Friends & Family: 0.10 (F1: 0.5763)
Optimal threshold for Grateful Patient: 0.10 (F1: 0.5769)
Optimal threshold for Health Education: 0.10 (F1: 0.7500)
Optimal threshold for Laboratory/Testing: 0.35 (F1: 0.8824)
Optimal threshold for Medications: 0.15 (F1: 0.9037)
Optimal threshold for No Symptoms: 0.50 (F1: 0.8030)
Optimal threshold for Non-urgent: 0.70 (F1: 0.9552)
Optimal threshold for Outpatient Logistics/Scheduling: 0.15 (F1: 0.6806)
Optimal threshold for Physical: 0.20 (F1: 0.7513)
Optimal threshold for Physical Environment/Climate: 0.10 (F1: 0.8421)
Optimal threshold for Service Complaint: 0.05 (F1: 0.3030)
Optimal threshold for Social Services: 0.15 (F1: 0.8333)
Optimal threshold for Technical/IT: 0.10 (F1: 0.7368)
Optimal threshold for Urgent: 

In [54]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

Y_pred_all = np.zeros(y_test.shape)

print("--- Individual Class F1 Scores ---")

for i, label_name in enumerate(label_keep_original):

    x_test_predic = models[label_name].predict_proba(x_test)[:, 1]

    y_test_final = (x_test_predic > optimal_thresholds[label_name]).astype(int)

    Y_pred_all[:, i] = y_test_final

    f1 = f1_score(y_test[:, i], y_test_final)
    count = dfCombined_filtered[label_name].sum()
    print(f"  {label_name:<40} F1: {f1:.4f}   Count: {count}")

print("\n" + "="*50 + "\n")

macro_f1    = f1_score(y_test, Y_pred_all, average='macro',    zero_division=0)
micro_f1    = f1_score(y_test, Y_pred_all, average='micro',    zero_division=0)
weighted_f1 = f1_score(y_test, Y_pred_all, average='weighted', zero_division=0)

print("--- Overall Test Set Metrics ---")
print(f"  Macro F1 Score:    {macro_f1:.4f}")
print(f"  Micro F1 Score:    {micro_f1:.4f}")
print(f"  Weighted F1 Score: {weighted_f1:.4f}")

print("--- Full Classification Report ---")
print(classification_report(
    y_test,
    Y_pred_all,
    target_names=label_keep_original,
    zero_division=0
))

--- Individual Class F1 Scores ---
  Cultural/Religion                        F1: 0.9159   Count: 343
  Diet/Nutrition                           F1: 0.8511   Count: 340
  Emergent                                 F1: 0.4286   Count: 140
  Exercise                                 F1: 0.7368   Count: 131
  Friends & Family                         F1: 0.6000   Count: 291
  Grateful Patient                         F1: 0.5593   Count: 291
  Health Education                         F1: 0.7312   Count: 322
  Laboratory/Testing                       F1: 0.9213   Count: 1078
  Medications                              F1: 0.8718   Count: 418
  No Symptoms                              F1: 0.8000   Count: 1658
  Non-urgent                               F1: 0.9526   Count: 3483
  Outpatient Logistics/Scheduling          F1: 0.7516   Count: 448
  Physical                                 F1: 0.7735   Count: 628
  Physical Environment/Climate             F1: 0.8649   Count: 203
  Service Complaint     

In [55]:

import numpy as np
from sklearn.metrics import f1_score

N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)
n = len(y_test)

boot_scores = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)
    score = f1_score(y_test[idx], Y_pred_all[idx], average='macro', zero_division=0)
    boot_scores.append(score)

boot_scores = np.array(boot_scores)
lo = np.percentile(boot_scores, 2.5)
hi = np.percentile(boot_scores, 97.5)
print(f"Macro F1: {macro_f1:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Macro F1: 0.6945  95% CI [0.6623, 0.7201]


In [57]:

import numpy as np, os
os.makedirs("predictions", exist_ok=True)



np.savez_compressed(
    "predictions/RandomForest_preds.npz",
    y_true = y_test,
    y_pred = Y_pred_all,
)
print("Saved predictions/RandomForest_preds.npz")


Saved predictions/RandomForest_preds.npz


## Per-corpus + per-label evaluation (RandomForest)

Loads saved `RandomForest_preds.npz` and reports Macro F1 (+ 95% bootstrap CI) and per-label F1 separately for the pooled corpus and for the Rwanda and Canada halves. Source labels come from `Ensemble_test_predictions.json`, which carries the per-row `source_corpus` field for the same 584-doc test set.


In [ ]:
# PERCORPUSCELL_v1
# Per-corpus + per-label evaluation against the shared test split, with 1,000-resample bootstrap CIs.
import json, numpy as np
from sklearn.metrics import f1_score, classification_report

with open("predictions/Ensemble_test_predictions.json") as f:
    _ens = json.load(f)
_src    = np.array([r["source_corpus"] for r in _ens])
_labels = list(_ens[0]["true_labels"].keys())

_d = np.load("predictions/RandomForest_preds.npz")
_yt, _yp = _d["y_true"].astype(int), _d["y_pred"]
if _yp.dtype != int:
    _yp = (_yp > 0.5).astype(int) if _yp.max() <= 1 else _yp.astype(int)

for _corpus in ["Combined", "Rwanda", "Canada"]:
    _mask = np.ones(len(_src), bool) if _corpus == "Combined" else (_src == _corpus)
    _yti, _ypi = _yt[_mask], _yp[_mask]
    _macro = f1_score(_yti, _ypi, average="macro", zero_division=0)

    _rng = np.random.default_rng(42)
    _n = len(_yti)
    _boot = np.empty(1000)
    for _i in range(1000):
        _idx = _rng.integers(0, _n, _n)
        _boot[_i] = f1_score(_yti[_idx], _ypi[_idx], average="macro", zero_division=0)
    _lo, _hi = np.percentile(_boot, [2.5, 97.5])

    print(f"\n=== {{_corpus}} (n={{_n}})  Macro F1 {{_macro:.4f}}  95% CI [{{_lo:.4f}}, {{_hi:.4f}}] ===")
    print(classification_report(_yti, _ypi, target_names=_labels, zero_division=0))
